# Soustraction Spectrale
### Outils
- Audacity
- MUSDB18-HQ (Format: .wav ; Encodage: signed 16-bit PCM) compatible avec soundfile (cf https://libsndfile.github.io/libsndfile/formats.html)

### Remarques

- Nous travaillons avec des fréquences d'echantillonage 44100 Hz equivalents aux CD (cf MUSDB18)
- Nous fixons la graine (0) afin que les resultats soient reproductibles
- Les DSP sont en decibels dB pour le reste de ce notebook (entrees et sorties)
### Importation des librairies et fichiers audio

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import soundfile as sf
from IPython.display import Audio, display
from scipy.signal import stft, istft
#from ..utils.tools import listen
np.random.seed(0)
fs_CD=44100

In [ ]:
#Import des fichiers sonores et verification de l'echantillonage
son1,samplerate=sf.read('../../data/son1.wav')
print('Echantillonage 1 :',samplerate)
son2,samplerate=sf.read('../../data/son2.wav')
print('Echantillonage 2 :',samplerate)
son3,samplerate=sf.read('../../data/son3.wav')
print('Echantillonage 3 :',samplerate)

### Fonctions de mesure et visualisation

In [ ]:
def SNR(u,v):
    Pu,Pv=np.sum(u**2)/len(u),np.sum(v**2)/len(v)
    return 10*np.log10(Pu/Pv)

def signal(x,fs,show=False):
    '''
    Retourne le signal avec son axe temporelle
    --- In ---
    x : signal (ndarray)
    fs : frequence d'echantillonage (float)
    show : trace le signal temporel (bool)
    --- Out ---
    t : axe temporel (ndarray)
    x : signal (ndarray)
    '''
    t=np.arange(len(x))/fs
    if show:
        plt.plot(t,x)
        plt.title('Signal')
        plt.ylabel('Amplitude')
        plt.xlabel('Time [sec]')
        plt.show()
    return t,x

def signal_power(x):
    return np.sum(x**2)/len(x)

def periodogram(x, fs: float,show=False):
    '''
    Periodogramme d'un signal entree
    --- In ---
    x : signal (ndarray)
    fs : frequence d'echantillonage (float)
    show : trace le periodogramme (bool)
    --- Out ---
    f : axe frequentiel (ndarray)
    Pgram : Densite spectrale DSP (ndarray)
    '''
    x_hat=np.fft.fft(x)
    f=np.fft.fftfreq(len(x),1/fs)
    Pgram=(np.abs(x_hat)**2)/(len(x)*fs)
    if show:
        plt.plot(f,Pgram,label='Spectral Density')
        plt.title('Periodogram')
        plt.ylabel('Spectral Density')
        plt.xlabel('Frequency [Hz]')
        plt.legend()
        plt.show()
    return f, Pgram
    
def spectrogram(x, fs: float, show=False):
    '''
    Trace le spectrogramme du signal d'entree
    --- In ---
    x : signal (ndarray)
    fs : frequence d'echantillonage (float)
    show : trace le periodogramme (bool)
    --- Out ---
    t : axe temporel
    f : axe frequentiel
    Sgram : de
    '''
    f,t,Sgram=stft(x,fs,window='hann',nperseg=1024,noverlap=256,boundary="zeros",padded=True,)
    if show:
        plt.pcolormesh(t, f, np.abs(Sgram))
        plt.title('STFT Magnitude')
        plt.ylabel('Frequency [Hz]')
        plt.xlabel('Time Index')
        plt.colorbar()
        plt.show()
    return t,f,Sgram



def white_noise(N: int, fs: float, mu: float, sigma: float,seed=0,show=False):
    '''
    Cree un signal 'bruit blanc'
    --- In ---
    N : longueur du signal (int)
    fs : frequence d'echantillonage (float)
    mu : moyenne du bruit (float)
    sigma : ecart-type du bruit (float)
    seed : graine np.random (int)
    --- Out ---
    noise : signal bruite (ndarray)
    '''
    np.random.seed(seed)
    noise=np.random.normal(mu,sigma,N)
    if show:
        signal(noise,fs,True)
    return noise

def add_white_noise(x, fs: float, target_SNR: float, seed=0, show=False):
    '''
    Ajoute du bruit blanc au signal jusqu'a obtenir la DSP totale recherchee.
    --- In ---
    x : signal (ndarray)
    fs : frequence d'echantillonage (float)
    target_SNR : SNR recherchee en sortie en dB (float)
    show : trace le signal en sortie (bool)
    --- Out ---
    x_b : signal bruite (ndarray)
    target_sigma : ecart type du bruit (float)
    '''
    target_sigma=(signal_power(x)*10**-(target_SNR/10))**.5
    x_b=x+white_noise(len(x),fs,0,target_sigma,seed,False)
    if show:
        signal(x_b,fs,True)
    return x_b,target_sigma

def listen(audio,samplerate):
    '''
    Affiche le lecteur sonore pour le signal d'entree
    --- In ---
    audio : signal (ndarray)
    samplerate : frequence d'echantillonage (float)
    --- Out ---
    None
    '''
    display(Audio(audio, rate=samplerate))
    return

def gSNR(u,v):
    Pu,Pv=np.sum(u**2)/len(u),np.sum(v**2)/len(v)
    return 10*np.log10(Pu/Pv)

def SNR(x,xd):
    '''
    Renvoie le SNR pour le signal pur et debruite
    --- In ---
    x : signal pur (ndarray)
    xd : signal debruite (ndarray)
    --- Out ---
    None
    '''
    max_len=min(len(x),len(xd))
    Px,Pxd=np.sum(x[:max_len]**2)/max_len,np.sum((xd[:max_len]-x[:max_len])**2)/max_len
    return 10*np.log10(Px/Pxd)

### Mise en place des donnees

#### Bruit Blanc

In [ ]:
white_1=white_noise(10000,fs_CD,0,10)
signal(white_1,fs_CD,True)

In [ ]:
#Visualisation de la Periode
f1,Pgram_1=periodogram(white_1,fs_CD,True)

In [ ]:
#Visualisation du Spectre
t,f,Sgram_b=spectrogram(white_1,fs_CD,True)

### Signal 1 - 

In [ ]:
#Visualisations du signal propre
t1,x1=signal(son1,fs_CD,True)
f1,Pgram1=periodogram(son1,fs_CD,True)
t1,f1,Sgram1=spectrogram(son1,fs_CD,True)

In [ ]:
#Visualisations du signal bruite blanc
tSNR=1
son1_b,s1=add_white_noise(son1,fs_CD,tSNR,0,True)
print(s1)
f2,Pgram2=periodogram(son1_b,fs_CD,True)
t2,f2,Sgram2=spectrogram(son1_b,fs_CD,True)


In [ ]:
#---Ne pas executer temporairement---
#Ecriture du son bruite
#sf.write('../../data/son1 bruite.wav',son1_b,44100)

### Signal 2 - 

In [ ]:
#Visualisations du signal propre
t1,x1=signal(son2,fs_CD,True)
f1,Pgram1=periodogram(son2,fs_CD,True)
t1,f1,Sgram1=spectrogram(son2,fs_CD,True)

In [ ]:
#Visualisations du signal bruite blanc
tSNR=1
son2_b,s2=add_white_noise(son2,fs_CD,tSNR,0,True)
print(s2)
f2,Pgram2=periodogram(son2_b,fs_CD,True)
t2,f2,Sgram2=spectrogram(son2_b,fs_CD,True)

In [ ]:
#---Ne pas executer temporairement---
#Ecriture du son bruite
#sf.write('../../data/son2 bruite.wav',son2_b,44100)

### Signal 3 -

In [ ]:
#Visualisations du signal propre
t1,x1=signal(son3,fs_CD,True)
f1,Pgram1=periodogram(son3,fs_CD,True)
t1,f1,Sgram1=spectrogram(son3,fs_CD,True)

In [ ]:
#Visualisations du signal bruite blanc
tSNR=1
son3_b,s3=add_white_noise(son3,fs_CD,tSNR,0,True)
print(s3)
f2,Pgram2=periodogram(son3_b,fs_CD,True)
t2,f2,Sgram2=spectrogram(son3_b,fs_CD,True)


In [ ]:
def denoise_spectral_sub(y, sigma, threshold, fs):
    '''
    Retourne le signal apres soustraction spectrale
    --- In ---
    y : signal (ndarray)
    sigma : ecart type du bruit (float)
    threshold : seuil de coupure (float)
    fs : frequence d'echantillonage (float)
    --- Out ---
    x : signal apres soustraction spectrale (ndarray)
    '''
    y_t,y_f,y_Sgram=spectrogram(y,fs)
    mask=np.abs(y_Sgram)>=threshold*sigma**2
    y_Sgram_mask= y_Sgram*mask
    t_denoise,y_denoise=istft(y_Sgram_mask,fs,window='hann',nperseg=1024,noverlap=256,boundary=True)
    return t_denoise,y_denoise


In [ ]:
threshold_list=[.5,.8,1,2,3]
sons_bruites=[son1_b,son2_b,son3_b]
sigmas=[s1,s2,s3]
for i in range(3):
    print('--- SON',i+1)
    for thresh in threshold_list:
        print('seuil :',thresh)
        _,son_debruite=signal(denoise_spectral_sub(sons_bruites[i],sigmas[i],thresh,fs_CD)[1],fs_CD,False)
        display(Audio(son_debruite, rate=fs_CD))


## Soustraction Spectrale du bruit blanc
#### Observations
En ecoutant les signaux debruites, on entend des artefacts auditifs pour des seuils importants et des restes de bruit blanc pour des seuils faibles.
Conclusion : Meme en connaissant la forme du bruit, une soustraction spectrale generale cree des artefacts qui ne sont pas plaisants a ecouter.

## Seuil Oracle
### Son1

In [ ]:
sigmas=[0.05, 0.1, 0.2, 0.5]
thresholds=np.linspace(0,5,100)

for sig in sigmas:
    oSNR_list=[]
    np.random.seed(0)
    son_b=np.array(son1+np.random.normal(0,sig,len(son1)))
    for thresh in thresholds:
        _,son_d=denoise_spectral_sub(son_b,sig,thresh,fs_CD)
        #max_len=min(len(son1),len(son_d))
        oSNR=SNR(son1,son_d)
        oSNR_list.append(oSNR)
    plt.plot(thresholds,oSNR_list,label='sigma='+str(sig))
plt.legend()
plt.title('Recherche du Seuil Oracle - son1')
plt.ylabel('SNR')
plt.xlabel('Threshold')
plt.grid()
plt.show()

#### Son2

In [ ]:
sigmas=[0.05, 0.1, 0.2, 0.5]
thresholds=np.linspace(0,5,100)

for sig in sigmas:
    oSNR_list=[]
    np.random.seed(0)
    son_b=np.array(son2+np.random.normal(0,sig,len(son2)))
    for thresh in thresholds:
        _,son_d=denoise_spectral_sub(son_b,sig,thresh,fs_CD)
        #max_len=min(len(son2),len(son_d))
        oSNR=SNR(son2,son_d)
        oSNR_list.append(oSNR)
    plt.plot(thresholds,oSNR_list,label='sigma='+str(sig))
plt.legend()
plt.title('Recherche du Seuil Oracle - son2')
plt.ylabel('SNR')
plt.xlabel('Threshold')
plt.grid()
plt.show()

#### Son3

In [ ]:
sigmas=[0.05, 0.1, 0.2, 0.5]
thresholds=np.linspace(0,5,100)

for sig in sigmas:
    oSNR_list=[]
    np.random.seed(0)
    son_b=np.array(son3+np.random.normal(0,sig,len(son3)))
    for thresh in thresholds:
        _,son_d=denoise_spectral_sub(son_b,sig,thresh,fs_CD)
        #max_len=min(len(son3),len(son_d))
        oSNR=SNR(son3,son_d)
        oSNR_list.append(oSNR)
    plt.plot(thresholds,oSNR_list,label='sigma='+str(sig))
plt.legend()
plt.title('Recherche du Seuil Oracle - son3')
plt.ylabel('SNR')
plt.xlabel('Threshold')
plt.grid()
plt.show()

#### Seuil oracle pour nos trois fichiers sonores
En applicant un bruit gaussien jusqu'a obtenir une SNR de 1 pour chaque fichier sonore, nous trouvons que les ecarts des trois bruits sont environs $\sigma_1=0.04$, $\sigma_2=0.1$, $\sigma_3=0.1$. A l'aide de notre graphe de seuil oracle on trouve que les seuils respectives correspondants sont 1.5, 1 et 1.

In [ ]:
print('Oracle Son1')
_,son1_oracle=signal(denoise_spectral_sub(sons_bruites[0],s1,2,fs_CD)[1],fs_CD,False)
listen(son1_oracle,fs_CD)
print('Oracle Son2')
_,son2_oracle=signal(denoise_spectral_sub(sons_bruites[1],s2,1,fs_CD)[1],fs_CD,False)
listen(son2_oracle,fs_CD)
print('Oracle Son3')
_,son3_oracle=signal(denoise_spectral_sub(sons_bruites[2],s3,1,fs_CD)[1],fs_CD,False)
listen(son3_oracle,fs_CD)

## Conclusion:
Nous remarquons que meme pour le seuil oracle, les artefacts sonores sont nuisants a l'ecoute meme si le son pure peut etre entendu. En particulier, nous notons des artefacts sonores du type clics et bips.

## Automatisation de la recherche du seuil oracle

In [ ]:
def find_oracle_threshold(x,xb,sigma,fs,denoise_func):
    '''
    Retourne le seuil oracle 
    --- In ---
    x : signal pur (ndarray)
    xb : signal bruite (ndarray)
    sigma : ecart type du bruit gaussien
    fs : frequence d'echantillonage (float)
    denoise_func : fonction de debruitage sous forme (y, sigma, threshold, fs) --> (t_denoise, y_denoise)
    --- Out ---
    oracle_threshold : seuil oracle du signal (float)
    '''
    thresholds=np.linspace(0,3,100)
    oSNR_max=-np.inf
    oracle_threshold=0
    for thresh in thresholds:
        _,x_d=denoise_func(xb,sigma,thresh,fs)
        max_len=min(len(x),len(x_d))
        oSNR=SNR(x,x_d)
        if oSNR>=oSNR_max:
            oSNR_max=oSNR
            oracle_threshold=thresh
    return oracle_threshold
    
    

In [ ]:
oracle1=find_oracle_threshold(son1,son1_b,s1,fs_CD,denoise_spectral_sub)
print('Oracle pour le son1 :',oracle1)

In [ ]:
print('Oracle Son1 Trouve')
_,son1_oracle=signal(denoise_spectral_sub(sons_bruites[0],s1,oracle1,fs_CD)[1],fs_CD,False)
listen(son1_oracle,fs_CD)

In [ ]:
oracle2=find_oracle_threshold(son2,son2_b,s2,fs_CD,denoise_spectral_sub)
print('Oracle pour le son2 :',oracle2)

In [ ]:
print('Oracle Son2 Trouve')
_,son2_oracle=signal(denoise_spectral_sub(sons_bruites[1],s2,oracle2,fs_CD)[1],fs_CD,False)
listen(son2_oracle,fs_CD)

In [ ]:
oracle3=find_oracle_threshold(son3,son3_b,s3,fs_CD,denoise_spectral_sub)
print('Oracle pour le son3 :',oracle3)

In [ ]:
print('Oracle Son3 Trouve')
_,son3_oracle=signal(denoise_spectral_sub(sons_bruites[2],s3,oracle3,fs_CD)[1],fs_CD,False)
listen(son3_oracle,fs_CD)